# Lab 1: Write / Externalize — Storing Knowledge Outside Prompts\n\n**Concept:** Instead of stuffing everything into the prompt, store knowledge in external structures (files, TF-IDF indexes, vector databases) and retrieve only what's needed. This reduces token cost and keeps prompts clean.\n\n## Exercises\n- 1a. Build a file-based external knowledge base that loads .txt files matching question keywords (from `pruning_dashboard`).\n- 1b. Precompute a TF-IDF index for document chunks and query it without reprocessing (from `semantic_chunk`).\n- 1c. Use an LLM to summarize a document into a compact 'knowledge card' stored externally (from `llmlingua`).\n

## Setup\n\n```bash\npip install openai scikit-learn numpy\nexport OPENROUTER_API_KEY='sk-or-v1-...'\n```

In [ ]:
import os, re, json, pickle, glob\nimport numpy as np\nfrom openai import OpenAI\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nclient = OpenAI(api_key=os.getenv('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1')

## Exercises

In [ ]:
import os, re, pickle, glob
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

# === Exercise 1a: File-based external knowledge base ===
document = """The company reported strong Q3 earnings. Revenue grew by 15% year over year to $45.2 million.
The expansion of the Frankfurt server farm cost $14.73 million. This was partially offset by a $2.1 million
green energy tax rebate. Employee headcount remained stable at 1,200. Management raised full-year guidance
citing strong demand in European markets."""
os.makedirs("_docs", exist_ok=True)
with open("_docs/finance_q3.txt", "w") as f:
    f.write(document)

def load_external_kb(directory, question):
    keywords = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)}
    merged = []
    for fp in glob.glob(os.path.join(directory, "*.txt")):
        if any(k in os.path.basename(fp).lower() for k in keywords):
            with open(fp) as f:
                merged.append(f.read().strip())
    return '\n\n'.join(merged) if merged else "No matching files."

print("=== 1a: External KB ===")
print(load_external_kb("_docs", "What was the CAPEX?")[:120])

# === Exercise 1b: TF-IDF precomputed index ===
words = document.split()
chunks = [' '.join(words[i:i+8]) for i in range(0, len(words), 5)]
vect = TfidfVectorizer(ngram_range=(1,2))
chunk_vectors = vect.fit_transform(chunks)
pickle.dump({"vect": vect, "chunks": chunks, "vectors": chunk_vectors},
            open("_tfidf_index.pkl", "wb"))

idx = pickle.load(open("_tfidf_index.pkl", "rb"))
q_vec = idx["vect"].transform(["What offset the costs?"])
scores = cosine_similarity(q_vec, idx["vectors"]).flatten()
print("\n=== 1b: TF-IDF Index ===")
for i in np.argsort(scores)[::-1][:2]:
    print(f"  {idx['chunks'][i]}")

# === Exercise 1c: LLM knowledge card ===
resp = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct:free",
    messages=[{"role": "user", "content":
        f"Extract key facts (numbers, dates, entities) as a compact knowledge card:\n\n{document}"}],
    temperature=0,
)
print("\n=== 1c: Knowledge Card ===")
kb_card = resp.choices[0].message.content
print(kb_card[:200])

# Cleanup
import shutil
shutil.rmtree("_docs", ignore_errors=True)
os.remove("_tfidf_index.pkl")

## Summary\n\nInstead of stuffing everything into the prompt, store knowledge in external structures (files, TF-IDF indexes, vector databases) and retrieve only what's needed. This reduces token cost and keeps prompts clean.